# Function 5


In [1]:

import numpy as np
import matplotlib.pyplot as plt
import math
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from scipy.stats import qmc



### Data preparation

Here we prepare the initial data provided

In [2]:
input = np.load('./initial_inputs.npy')
output = np.load('./initial_outputs.npy')
print(input)
print(output)
#print(input.shape)
#print(input.shape[1])
func_dimensions = input.shape[1]
print('this function has ', func_dimensions, ' dimensions')

[[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]]
[6.44434399e+01 1.83013796e+01 1.12939795e-01 4.21089813e+0

Here we add the data provided with the weekly queries

In [3]:
additionalInputs = [[0.5, 0.5, 0.5, 0.5], [0.210195, 0.849014, 0.874271, 0.875537], [0.241981, 0.847887, 0.878175, 0.876911], [0.230056, 0.841143, 0.873217, 0.887638], [0.241981, 0.847887, 0.878175, 0.876911], [0.0, 0.24, 0.62, 1.0], [0.29, 0.99, 1.0, 1.0], [0.281347, 0.989493, 0.991107, 0.998584], [0.317606, 0.990506, 0.995215, 0.993607], [0.295741, 0.990445, 0.999436, 0.984022], [0.69, 1.0, 1.0, 1.0]]
additionalOutputs = [np.float64(32.0025), np.float64(1060.3898045443261), np.float64(1083.0108574790386), np.float64(1081.4793716878269), np.float64(1083.0108574790386), np.float64(321.73026824755283), np.float64(4333.689411920009), np.float64(4185.47205258832), np.float64(4201.38401252448), np.float64(4122.869521765046), np.float64(5300.350621253554)]

input = np.append(input, additionalInputs, axis=0)
output = np.append(output, additionalOutputs)

print(input)
print(output)

[[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]
 [0.5        0.5        0.5        0.5       ]
 [0.210195   

# Bayesian Optimisation approach
We approach the study of this function with the Bayesian Optimisation
using and adaptation of the UCB acquisition function from required assignment 12.1

# Preparation of the exploration space
Here we prepare the exploration space of the function<br>
The values are increments of 0.01 bounded between 0 and 1 (included) => 101 values for each dimension of the space<br>
dimension of the current function is stored in <b>func_dimensions</b>

In [4]:
#Initialise query lists and maximum observations
X, Y = input, output

# --- Latin Hypercube Sampling (LHS) ---
n_samples = 12500000
sampler_lhs = qmc.LatinHypercube(d=func_dimensions, seed=42)
x_grid = sampler_lhs.random(n=n_samples)
#print(f"LHS shape: {x_grid_lhs.shape}")

#x_grid = np.delete(x_grid, 0, axis=0)
#print(x_grid)
print(x_grid.shape)


(12500000, 4)


In [5]:
print(x_grid[:10])

[[0.07564682 0.19730556 0.61438713 0.9059265 ]
 [0.29252671 0.03016936 0.6251265  0.07986546]
 [0.79961855 0.61025564 0.27169621 0.86352841]
 [0.94375003 0.14954025 0.35439468 0.42597542]
 [0.00237004 0.12494751 0.09195369 0.27859139]
 [0.93379658 0.02449221 0.97306184 0.30644521]
 [0.05330282 0.53045086 0.98159828 0.84677248]
 [0.32496575 0.80649195 0.67306674 0.55027   ]
 [0.18394533 0.21741149 0.4257746  0.58111678]
 [0.76197927 0.99845388 0.94852926 0.00507875]]


# Bayesian Optimisation with UCB applied

In [6]:
rbf_lengthscale = [0.1, 0.1, 0.1, 0.1]
real_noise_std = 1e-10
noise_assumption = 1e-10

#Define kernel of GP
kernel = RBF(length_scale=rbf_lengthscale, length_scale_bounds='fixed')

model = GaussianProcessRegressor(kernel = kernel)
#Fit the model
model.fit(np.array(X), np.array(Y).reshape(-1, 1))


#Calculate the mean and standard deviation and make them one-dimensional for plotting
post_mean, post_std = model.predict(x_grid, return_std=True)

#Acquisition function parameter
#beta = 1.96
#beta = 0.5
#beta = 10
beta = 0.02

acquisition_function = post_mean + beta * post_std

grid = x_grid.squeeze()
obs = grid[np.argmax(acquisition_function)] #Else use the acquisition function

print('Next observation: ', obs)
#print (obs)


Next observation:  [0.67831057 0.98530093 0.99086977 0.99504031]


In [ ]:
np.set_printoptions(suppress=True)
combined = []

for i, v in enumerate(input):
  combined = np.append(combined, np.append(input[i], output[i]))
print(combined.reshape(-1, 5))

[[   0.19144708    0.03819337    0.60741781    0.41458414   64.44343986]
 [   0.75865295    0.53651774    0.65600038    0.36034155   18.3013796 ]
 [   0.43834987    0.8043397     0.21024527    0.15129482    0.1129398 ]
 [   0.70605083    0.53419196    0.26424335    0.48208755    4.21089813]
 [   0.83647799    0.19360965    0.6638927     0.78564888  258.37052545]
 [   0.68343225    0.11866264    0.82904591    0.56757661   78.43438889]
 [   0.55362148    0.66734998    0.32380582    0.81486975   57.57153693]
 [   0.35235627    0.32224153    0.11697937    0.47311252  109.57187556]
 [   0.15378571    0.72938169    0.42259844    0.44307417    8.84799176]
 [   0.46344227    0.63002451    0.10790646    0.9576439   233.22361017]
 [   0.67749115    0.35850951    0.47959222    0.07288048   24.42308831]
 [   0.58397341    0.14724265    0.34809746    0.42861465   64.42014682]
 [   0.30688872    0.31687813    0.62263448    0.09539906   63.47671579]
 [   0.51114177    0.817957      0.72871042    0.11

In [ ]:
#print(input)
np.corrcoef(input)

array([[ 1.        , -0.07567623, -0.88750938, -0.77416008,  0.54863557,
         0.81040184, -0.46602919, -0.43979835, -0.28103918, -0.37812522,
        -0.16939658,  0.25477302,  0.42262091, -0.25780109, -0.29878501,
         0.36652083, -0.50251064, -0.52352573,  0.09348703,  0.81776519,
                nan,  0.35652402,  0.36409809,  0.36941557,  0.36409809],
       [-0.07567623,  1.        ,  0.1940994 ,  0.2280302 ,  0.16052303,
         0.38085998, -0.7476116 , -0.57172642, -0.54456529, -0.78335275,
         0.99537106,  0.35833476,  0.63083396,  0.56425582, -0.92321773,
        -0.70645535, -0.2965377 ,  0.40101227, -0.83315827,  0.22358643,
                nan, -0.70793059, -0.70602488, -0.72063936, -0.70602488],
       [-0.88750938,  0.1940994 ,  1.        ,  0.43867381, -0.82034204,
        -0.83234429,  0.13301377,  0.01985127,  0.51993705,  0.03690431,
         0.26766062, -0.57194091, -0.01763216,  0.65126779,  0.09801215,
        -0.13103224,  0.05039826,  0.85224923,  0